# 04 - Qualidade de Dados
Persiste resultados objetivos para completude, consistência, unicidade, cobertura e outliers.

In [0]:
%run ./00_setup

# 00 - Configuração
Cria objetos do Unity Catalog e caminhos do MVP. Ajuste os widgets antes da primeira execução.

Envie COTAHIST_A2025.TXT para: /Volumes/workspace/mvp_b3/landing/cotahist/
Envie setores_b3.csv para: /Volumes/workspace/mvp_b3/landing/referencia/


In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone

q = spark.table(f"{catalog}.{schema}.silver_cotacoes")
f = spark.table(f"{catalog}.{schema}.fato_cotacao_diaria")
d = spark.table(f"{catalog}.{schema}.dim_ativo")

def count_where(df, expression):
    return df.filter(expression).count()

total = q.count()
q1, q3 = f.where("retorno_diario IS NOT NULL").approxQuantile("retorno_diario", [0.25, 0.75], 0.01)
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr

checks = [
    ("completude_chaves", "completude", count_where(q, "data_pregao IS NULL OR ticker IS NULL OR preco_fechamento IS NULL"), 0, "FAIL_IF_ABOVE"),
    ("precos_invalidos", "consistencia", count_where(q, "preco_fechamento <= 0 OR preco_maximo < preco_minimo"), 0, "FAIL_IF_ABOVE"),
    ("valores_negativos", "acuracia", count_where(q, "volume_financeiro < 0 OR quantidade_titulos < 0 OR numero_negocios < 0"), 0, "FAIL_IF_ABOVE"),
    ("duplicidade_chave", "unicidade", q.groupBy("data_pregao", "ticker").count().filter("count > 1").count(), 0, "FAIL_IF_ABOVE"),
    ("ativos_sem_setor", "cobertura", count_where(d, "setor = 'NAO_INFORMADO'"), 0, "WARN_IF_ABOVE"),
    ("retornos_outliers_iqr", "outliers", count_where(f, (F.col("retorno_diario") < lower) | (F.col("retorno_diario") > upper)), -1, "INFORMATIVO")
]

rows = []
for regra, dimensao, valor, limite, tipo in checks:
    status = "INFO" if tipo == "INFORMATIVO" else ("OK" if valor <= limite else ("ALERTA" if tipo == "WARN_IF_ABOVE" else "FALHA"))
    rows.append((datetime.now(timezone.utc), regra, dimensao, float(valor), float(limite), status, total))

result = spark.createDataFrame(rows, "execution_ts timestamp, regra string, dimensao string, valor double, limite double, status string, total_registros long")
(result.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{catalog}.{schema}.dq_resultados"))
display(result.orderBy("status", "regra"))

failures = result.filter("status = 'FALHA'").count()
if failures:
    raise ValueError(f"Pipeline bloqueado: {failures} regra(s) impeditiva(s) de qualidade falharam.")


execution_ts,regra,dimensao,valor,limite,status,total_registros
2026-09-23T17:28:57.815Z,retornos_outliers_iqr,outliers,7449.0,-1.0,INFO,85890
2026-09-23T17:28:57.815Z,ativos_sem_setor,cobertura,0.0,0.0,OK,85890
2026-09-23T17:28:57.815Z,completude_chaves,completude,0.0,0.0,OK,85890
2026-09-23T17:28:57.815Z,duplicidade_chave,unicidade,0.0,0.0,OK,85890
2026-09-23T17:28:57.815Z,precos_invalidos,consistencia,0.0,0.0,OK,85890
2026-09-23T17:28:57.815Z,valores_negativos,acuracia,0.0,0.0,OK,85890
